# Week 19 Optional (AWS / SageMaker) - Advanced MLOps Patterns

This is an OPTIONAL, async deep-dive notebook for students who finished the main Week 19 AWS / SageMaker notebook (`week_19_aws_di_mlops_versioning_experiments.ipynb`) and want to push their production-MLOps skills further.

The main notebook taught the happy path: track an experiment with MLflow, run a SageMaker Training Job, register the artifact in the SageMaker Model Package Registry, deploy an endpoint, and wire it back into the Week 18 supervisor.

This notebook goes deep on four topics the main notebook only touched on:

- **Topic 1 - pyfunc custom flavor**: Log a Strands-agent wrapper and a rule-based scorer as generic `mlflow.pyfunc` so any MLflow-compatible runtime can serve them.
- **Topic 2 - Nested runs, aliases, artifact logging**: Run a 3x3 hyperparameter sweep with `nested=True`, log a confusion-matrix PNG as a run artifact, pick the winner with `mlflow.search_runs()`, then assign the `production` alias.
- **Topic 3 - Local data versioning with DVC**: `dvc init`, `dvc add` a local copy of the batch, push to S3. Show that the `.dvc` pointer file is the only thing that lives in git.
- **Topic 4 - Lineage and SageMaker Pipelines preview**: Round-trip lineage via `CustomerMetadataProperties` and `list_associations`. See how the Week 19 training job becomes a `TrainingStep` inside a `Pipeline` with `RegisterModel` and `ConditionStep` for CI/CD.

## Prerequisites

- You finished `exercises/week_19_mlops/aws/week_19_aws_di_mlops_versioning_experiments.ipynb`.
- Same SageMaker Studio Lab user, same execution role, same shared S3 bucket (`bread-academy-week19-shared`).
- The KB id `FARSQGTONR` from Weeks 17/18 is available if you want to extend Lab 1 to ground the agent in the docs corpus.
- A registered Model Package ARN and a deployed endpoint name from the main notebook (you will look up both).

## Expected runtime

Roughly 60 minutes async. Skip any topic that does not interest you - each topic is self-contained after the setup cells.


## Environment Setup

**Platform**: AWS SageMaker Studio Lab in the `di-mfa` account (535146832369, us-east-1).

**Auth**: `sagemaker.Session()` + `get_execution_role()`. No `getpass`, no Databricks `dbutils.secrets`. Every AWS call uses the execution role attached to your Studio Lab user.

**S3 bucket**: `bread-academy-week19-shared` (shared across the class for Week 19 artifacts).

**New library this notebook**: `dvc[s3]` is the only new install beyond what the main Week 19 notebook used. You can also install it once from the System Terminal with `pip install "dvc[s3]"`, but for class convenience we install it from the notebook below.

**Bedrock LLM**: `us.anthropic.claude-sonnet-4-5-20250929-v1:0` (the only Bedrock model the class account has access to). The pre-flight probe in Cell 4 will fail loud if access is missing.

If you have not finished the main Week 19 notebook yet, stop and finish it first. This notebook reuses its tracking server, S3 bucket, and execution role.


In [ ]:
!pip install --quiet "mlflow==3.10.0" "sagemaker-mlflow>=0.1.0" "boto3>=1.35" "sagemaker==2.257.3" "strands-agents>=1.37,<2" "dvc[s3]>=3.50" "matplotlib>=3.7" "scikit-learn>=1.3"

import os
import json
import time
import uuid
import subprocess
from datetime import datetime
from importlib.metadata import version

import boto3
import pandas as pd
import mlflow

# Version check via importlib.metadata (never pkg.__version__)
for pkg in ["mlflow", "sagemaker-mlflow", "boto3", "sagemaker", "strands-agents", "dvc", "scikit-learn", "matplotlib"]:
    try:
        print(f"{pkg:25s} {version(pkg)}")
    except Exception as e:
        print(f"{pkg:25s} NOT INSTALLED ({e})")


In [ ]:
import sagemaker
from sagemaker import get_execution_role

# Canonical SageMaker-week auth block. No dbutils, no getpass.
sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

boto_sess = boto3.Session(region_name=AWS_REGION)
s3_client         = boto_sess.client("s3")
sagemaker_client  = boto_sess.client("sagemaker")
sts_client        = boto_sess.client("sts")
bedrock_runtime   = boto_sess.client("bedrock-runtime")

caller = sts_client.get_caller_identity()
print(f"Account: {caller['Account']}")
print(f"Role:    {role}")
print(f"Region:  {AWS_REGION}")

# Shared class bucket + per-student prefix so artifacts do not collide
S3_BUCKET = "bread-academy-week19-shared"
USER_ID   = caller["UserId"][:8]
S3_PREFIX = f"students/{USER_ID}/optional"
EXPERIMENT_NAME = f"week19-optional-{USER_ID}"

# MLflow tracking-server ARN from the main notebook
# (provided by instructor as an env var on the Studio Lab user)
MLFLOW_TRACKING_ARN = os.environ.get("MLFLOW_TRACKING_ARN", "")
print(f"MLflow ARN: {MLFLOW_TRACKING_ARN or '(set MLFLOW_TRACKING_ARN before continuing)'}")


In [ ]:
# Pre-flight probes: MLflow, S3, Bedrock LLM. Each fails LOUD with a
# specific message so you know exactly what to ask the instructor for.

# Guard: refuse to run if MLflow tracking ARN is missing - otherwise mlflow
# would silently fall back to a local file:// store and confuse everyone.
if not MLFLOW_TRACKING_ARN:
    raise RuntimeError(
        "MLFLOW_TRACKING_ARN is empty. Ask your instructor for the SageMaker "
        "MLflow tracking-server ARN and export it before running this notebook."
    )

mlflow.set_tracking_uri(MLFLOW_TRACKING_ARN)

# Probe 1: MLflow tracking server reachable
try:
    experiments = mlflow.search_experiments(max_results=1)
    print(f"MLflow OK: tracking_uri={mlflow.get_tracking_uri()}")
except Exception as e:
    print(f"MLflow FAIL: {e}\nAsk your instructor to confirm the SageMaker MLflow tracking server is running.")
    raise

# Probe 2: S3 bucket access
try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    print(f"S3 OK: s3://{S3_BUCKET}")
except Exception as e:
    print(f"S3 FAIL: {e}")
    raise

# Probe 3: Bedrock LLM (Haiku 3) - fail loud BEFORE any Topic 1 agent code runs
LLM_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
try:
    resp = bedrock_runtime.converse(
        modelId=LLM_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print(f"Bedrock OK: {LLM_MODEL_ID}")
except Exception as e:
    print(f"Bedrock FAIL: {e}\nAsk your instructor to enable Bedrock model access for {LLM_MODEL_ID}.")
    raise

# Set (or create) the experiment that all four topics will log into
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME}")


## Topic 1 - pyfunc custom flavor

MLflow ships built-in "flavors" for common frameworks: sklearn, pytorch, huggingface, langchain. The flavor handles serialization, dependencies, and the serving contract.

When you build something MLflow does not have a flavor for - a custom preprocessing pipeline, a multi-step chain, a Strands agent wrapper - you log it as a `pyfunc`. That is the universal flavor.

**Contract**:

- Subclass `mlflow.pyfunc.PythonModel`.
- Implement `predict(self, context, model_input, params=None)`.
- Optionally implement `load_context(self, context)` if you need to lazy-load heavy state (a boto3 client, a Bedrock agent, a model artifact from S3) only AFTER the artifact is deserialized. This is critical for objects whose internals are not pickle-safe.

Docs: `https://mlflow.org/docs/latest/ml/model/python_model/`

### Why this matters for Bread Financial

Your fraud team has a custom decision rule that combines the Week 14 DistilBERT output with a business-rules table. If you wrap that as a pyfunc, ops can serve it behind ANY MLflow-compatible runtime - a SageMaker endpoint, a local Docker container, a Lambda - without rewriting the inference code. The same `models:/<name>@production` URI works everywhere.


In [ ]:
import mlflow.pyfunc

class ThresholdAlertModel(mlflow.pyfunc.PythonModel):
    """Wraps a fraud score with a business-rules threshold and dollar cap.

    Inputs: pandas DataFrame with columns 'fraud_score' (float, 0..1) and
            'amount_usd' (float).
    Output: pandas DataFrame with columns 'action' (str) and 'reason' (str).
    """

    def __init__(self, score_threshold=0.7, amount_cap_usd=5000):
        # Parameters are pickle-safe scalars, so __init__ is fine here
        self.score_threshold = score_threshold
        self.amount_cap_usd = amount_cap_usd

    def predict(self, context, model_input, params=None):
        # Iterate the input DataFrame row by row and apply the business rules
        out = []
        for _, row in model_input.iterrows():
            if row["fraud_score"] >= self.score_threshold and row["amount_usd"] >= self.amount_cap_usd:
                out.append({"action": "block",   "reason": "high_score_high_amount"})
            elif row["fraud_score"] >= self.score_threshold:
                out.append({"action": "review",  "reason": "high_score"})
            else:
                out.append({"action": "approve", "reason": "low_score"})
        return pd.DataFrame(out)

# An input_example helps MLflow infer the signature and ships with the model
example_input = pd.DataFrame([
    {"fraud_score": 0.95, "amount_usd": 8000.0},
    {"fraud_score": 0.85, "amount_usd": 100.0},
    {"fraud_score": 0.20, "amount_usd": 2500.0},
])

with mlflow.start_run(run_name="topic1-demo-threshold-alert") as run:
    mlflow.log_params({"score_threshold": 0.7, "amount_cap_usd": 5000})
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ThresholdAlertModel(score_threshold=0.7, amount_cap_usd=5000),
        input_example=example_input,
    )
    print(f"Logged model URI: {info.model_uri}")
    DEMO_MODEL_URI = info.model_uri

# Load it back and run it to prove the round-trip contract works
loaded = mlflow.pyfunc.load_model(DEMO_MODEL_URI)
print(loaded.predict(example_input))


### Lab 1: Wrap a Strands agent as a pyfunc

Build a `StrandsFraudExplainer` `PythonModel` that:

1. Takes a single-column DataFrame with column `transaction_description`.
2. In `load_context()`, builds a Strands `Agent` with model `us.anthropic.claude-sonnet-4-5-20250929-v1:0` and a system prompt that asks the agent to explain in one sentence whether the transaction looks like fraud.
3. In `predict()`, iterates rows, calls `self._agent(text)` for each, and returns a DataFrame with two columns: `transaction_description` and `explanation`.
4. Logs the pyfunc with `mlflow.pyfunc.log_model()` using the example transaction strings below.
5. Loads it back and runs `loaded.predict(example_df)` to verify the round-trip.

Example transactions to use:

- `"Grocery purchase at Whole Foods - 84.32 USD"`
- `"ATM withdrawal in Lagos Nigeria - 1500 USD - card present"`
- `"Subscription renewal Netflix - 15.99 USD"`

**Important**: Store the agent on `self._agent` inside `load_context()` so it gets rebuilt after deserialization. Do NOT build the agent in `__init__()` - the boto3 client it wraps is not pickle-safe and the run will fail to log.

Also note: to actually serve this pyfunc on a SageMaker endpoint, the inference container would need `strands-agents`, `boto3`, and IAM permission to call Bedrock. We do not deploy it here; the goal is to prove the pyfunc round-trip.

**Success criterion**: `loaded.predict(example_df)` returns three rows with non-empty `explanation` strings.


In [ ]:
from strands import Agent

class StrandsFraudExplainer(mlflow.pyfunc.PythonModel):
    """Wrap a Strands agent so it can be served as an MLflow pyfunc."""

    def __init__(self, model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0", system_prompt=None):
        self.model_id = model_id
        self.system_prompt = system_prompt or (
            "You are a fraud analyst. In one sentence, explain whether the "
            "following transaction looks suspicious and why."
        )

    def load_context(self, context):
        # YOUR CODE
        self._agent = None

    def predict(self, context, model_input, params=None):
        rows = None  # YOUR CODE
        return pd.DataFrame(rows)

example_df = pd.DataFrame({
    "transaction_description": [
        "Grocery purchase at Whole Foods - 84.32 USD",
        "ATM withdrawal in Lagos Nigeria - 1500 USD - card present",
        "Subscription renewal Netflix - 15.99 USD",
    ]
})

LAB1_MODEL_URI = None  # YOUR CODE

with mlflow.start_run(run_name="topic1-lab1-strands-explainer") as run:
    info = None  # YOUR CODE
    print("Model URI:", LAB1_MODEL_URI)

if LAB1_MODEL_URI is not None:
    loaded = mlflow.pyfunc.load_model(LAB1_MODEL_URI)
    result = loaded.predict(example_df)
    print(result)


In [ ]:
# Lab 1 SAFETY-NET: run this only if you skipped or could not finish Lab 1.
# Skip this cell if your LAB1_MODEL_URI is set and the previous cell printed
# three explanation rows.
if LAB1_MODEL_URI is None:
    print("Using Lab 1 safety-net so Topic 2 still has a working pyfunc.")

    class StrandsFraudExplainer(mlflow.pyfunc.PythonModel):
        def __init__(self, model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0", system_prompt=None):
            self.model_id = model_id
            self.system_prompt = system_prompt or (
                "You are a fraud analyst. In one sentence, explain whether the "
                "following transaction looks suspicious and why."
            )

        def load_context(self, context):
            from strands import Agent
            self._agent = Agent(model=self.model_id, system_prompt=self.system_prompt)

        def predict(self, context, model_input, params=None):
            rows = []
            for _, row in model_input.iterrows():
                text = row["transaction_description"]
                msg = self._agent(text)
                rows.append({"transaction_description": text, "explanation": str(msg)})
            return pd.DataFrame(rows)

    with mlflow.start_run(run_name="topic1-lab1-safetynet") as run:
        info = mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=StrandsFraudExplainer(),
            input_example=example_df,
        )
        LAB1_MODEL_URI = info.model_uri
        print("Safety-net URI:", LAB1_MODEL_URI)


## Topic 2 - Nested runs, artifacts, aliases, search

Hyperparameter sweeps create many runs. MLflow has first-class support for the parent/child pattern:

- `mlflow.start_run(nested=True)` inside an outer `with mlflow.start_run():` block creates a child run. The MLflow UI shows children indented under the parent so a 3x3 grid renders as one parent + nine children.
- `mlflow.log_artifact(local_path)` saves any file (image, JSON, CSV) under the run's artifact root in S3. The canonical use is a confusion matrix PNG.
- `mlflow.search_runs(experiment_ids=[exp_id], filter_string=..., order_by=[...])` returns a pandas DataFrame, filterable by metric, param, or tag.
- Model aliases (MLflow 2.13+) replace the deprecated "stage" concept. After picking a winner you call `client.set_registered_model_alias(name, alias="production", version=N)`. Production code then loads with `mlflow.pyfunc.load_model("models:/<name>@production")` and you can swap the alias to point at a new version without changing any consumer code.

**Verified syntax** (from `https://mlflow.org/docs/latest/ml/search/search-runs/`):

- Filter: `'metrics.f1 > 0.8 and params.lr = "0.001"'`. AND only. No OR.
- `order_by` takes a list: `["metrics.f1 DESC"]`.

**Gotcha on managed MLflow** (the SageMaker MLflow plugin): the artifact root is the S3 bucket configured at tracking-server creation. Files logged via `log_artifact` land there automatically - you do not need to set `tracking_uri` per-artifact.


In [ ]:
import random
import tempfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

EXP_ID = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

def fake_train_and_predict(lr, batch_size, n=200, seed=0):
    """A toy 'training' function: bigger distance from the sweet-spot lr -> worse skill."""
    rng = np.random.default_rng(seed)
    distance = abs((lr - 0.001) / 0.001)
    skill = max(0.0, 0.92 - 0.15 * distance)
    y_true = rng.integers(0, 2, n)
    flip = rng.random(n) > skill
    y_pred = np.where(flip, 1 - y_true, y_true)
    tp = float(((y_pred == 1) & (y_true == 1)).sum())
    denom = max(1.0, (y_pred == 1).sum() + (y_true == 1).sum() - tp)
    f1 = tp / denom
    return y_true, y_pred, {"f1": f1, "loss": 1 - skill}

def log_confusion_matrix_png(y_true, y_pred, name):
    """Render a 2x2 confusion matrix PNG and log it as a run artifact under plots/."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    ax.set_title(name)
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
        fig.savefig(f.name, bbox_inches="tight")
        plt.close(fig)
        mlflow.log_artifact(f.name, artifact_path="plots")

sweep_grid = [
    {"lr": 0.01,   "batch_size": 32},
    {"lr": 0.001,  "batch_size": 32},
    {"lr": 0.0001, "batch_size": 32},
    {"lr": 0.001,  "batch_size": 64},
]

# Capture parent_run_id so the next cell can filter children to THIS sweep only
# and avoid picking up stale runs from a prior notebook execution.
LR_DEMO_PARENT_RUN_ID = None

with mlflow.start_run(run_name="topic2-demo-lr-sweep") as parent:
    LR_DEMO_PARENT_RUN_ID = parent.info.run_id
    mlflow.set_tag("sweep_kind", "lr_demo")
    mlflow.log_param("grid_size", len(sweep_grid))
    for i, hp in enumerate(sweep_grid):
        with mlflow.start_run(run_name=f"child-{i}-lr{hp['lr']}", nested=True):
            mlflow.log_params(hp)
            y_true, y_pred, metrics = fake_train_and_predict(seed=i, **hp)
            mlflow.log_metrics(metrics)
            mlflow.set_tag("child_index", str(i))
            mlflow.set_tag("sweep_kind", "lr_demo")
            log_confusion_matrix_png(y_true, y_pred, f"child-{i}")
            print(f"  child {i}: lr={hp['lr']:.4f}  f1={metrics['f1']:.4f}")
    print(f"Parent run id: {LR_DEMO_PARENT_RUN_ID}")


In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

# Search across the children of THIS sweep only. Filtering by sweep_kind alone
# would pick up stale children from a previous notebook run; combining with
# parentRunId keeps the result deterministic across re-executions.
runs_df = mlflow.search_runs(
    experiment_ids=[EXP_ID],
    filter_string=(
        'tags.sweep_kind = "lr_demo" '
        f'and tags.mlflow.parentRunId = "{LR_DEMO_PARENT_RUN_ID}"'
    ),
    order_by=["metrics.f1 DESC"],
    max_results=50,
)
sweep_children = runs_df[runs_df["metrics.f1"].notna()].copy()
print(f"Found {len(sweep_children)} child runs")
if sweep_children.empty:
    raise RuntimeError(
        "No lr_demo child runs found for the current parent run. "
        "Re-run the previous sweep cell before this one."
    )
print(sweep_children[["run_id", "tags.mlflow.runName", "params.lr", "params.batch_size", "metrics.f1"]])

best = sweep_children.iloc[0]
print(f"\nBest run_id: {best['run_id']} with f1={best['metrics.f1']:.4f}")

# Register the threshold pyfunc from the Topic 1 demo as a registered model,
# then assign the `production` alias.
# (The children in this lr demo did not log a model artifact - they only logged
# metrics - so we fall back to DEMO_MODEL_URI to demonstrate the alias call.)
REG_NAME = f"week19-optional-threshold-{USER_ID}"
try:
    reg = mlflow.register_model(model_uri=DEMO_MODEL_URI, name=REG_NAME)
    print(f"Registered: {REG_NAME} version {reg.version}")
    client.set_registered_model_alias(name=REG_NAME, alias="production", version=reg.version)
    print(f"Alias `production` -> version {reg.version}")
    loaded_prod = mlflow.pyfunc.load_model(f"models:/{REG_NAME}@production")
    print("Loaded by alias OK.")
except Exception as e:
    print(f"Register/alias step skipped: {e}")


### Lab 2: Threshold sweep with confusion-matrix artifact + production alias

In Lab 1 you wrapped a fraud explainer agent. The threshold pyfunc from the Topic 1 demo (`ThresholdAlertModel`) has two hyperparameters: `score_threshold` and `amount_cap_usd`. You want to find the combination that maximizes a "captured fraud value" metric on a held-out batch, save a confusion-matrix PNG per child run, and promote the winner to the `production` alias.

Steps:

1. Use the provided `BATCH_DF` (100 synthetic transactions with `fraud_score`, `amount_usd`, and `is_fraud` ground truth).
2. Run a sweep over a 3x3 grid of `score_threshold` in `[0.5, 0.7, 0.9]` and `amount_cap_usd` in `[1000, 3000, 5000]`.
3. For each grid cell, log a NESTED run with:
   - **params**: `score_threshold`, `amount_cap_usd`.
   - **metrics**: `captured_fraud_usd` (sum of `amount_usd` over rows where the model said `block` AND `is_fraud == 1`), `false_block_usd` (sum where the model said `block` AND `is_fraud == 0`).
   - **artifact**: confusion matrix PNG. Use the helper `log_confusion_matrix_png` from the demo with `y_true = is_fraud` and `y_pred = (action == "block").astype(int)`.
   - **tag**: `sweep_kind = "threshold_lab"`.
4. After the sweep, call `mlflow.search_runs()` filtered by that tag, rank by `captured_fraud_usd / (1 + false_block_usd)`, and pick the winner.
5. Build a fresh `ThresholdAlertModel` with the winning params, log it as a new pyfunc, register it under `LAB2_REG_NAME`, and assign the `production` alias to that version.
6. Load by alias and run one smoke prediction to confirm the alias resolves.

**Success criterion**: the winner print shows both hyperparameters, the `search_runs` DataFrame has exactly 9 child rows tagged `threshold_lab`, and `mlflow.pyfunc.load_model("models:/<name>@production")` returns a usable model.


In [ ]:
np.random.seed(0)
N = 100
BATCH_DF = pd.DataFrame({
    "fraud_score": np.clip(np.random.beta(2, 5, N) + np.random.normal(0, 0.1, N), 0, 1),
    "amount_usd":  np.random.gamma(2.0, 800, N).round(2),
})
BATCH_DF["is_fraud"] = (
    (BATCH_DF["fraud_score"] > 0.6) & (BATCH_DF["amount_usd"] > 500)
).astype(int)

print(f"Batch size: {len(BATCH_DF)}, fraud rate: {BATCH_DF['is_fraud'].mean():.2%}")

threshold_grid = [0.5, 0.7, 0.9]
cap_grid       = [1000, 3000, 5000]

LAB2_PARENT_RUN_ID = None
LAB2_REG_NAME = f"week19-optional-lab2-{USER_ID}"

with mlflow.start_run(run_name="topic2-lab2-threshold-sweep") as parent:
    mlflow.set_tag("sweep_kind", "threshold_lab")
    LAB2_PARENT_RUN_ID = parent.info.run_id

    for thr in threshold_grid:
        for cap in cap_grid:
            with mlflow.start_run(run_name=f"thr{thr}-cap{cap}", nested=True):
                # YOUR CODE
                pass

# YOUR CODE
runs_df = None
winner = None
print("Winner:", winner)

# YOUR CODE
# Register the winning model and assign the `production` alias.
LAB2_PROD_VERSION = None


In [ ]:
# Lab 2 SAFETY-NET: run only if you did not finish Lab 2.
if LAB2_PARENT_RUN_ID is None or winner is None or LAB2_PROD_VERSION is None:
    print("Using Lab 2 safety-net.")
    with mlflow.start_run(run_name="topic2-lab2-safetynet") as parent:
        mlflow.set_tag("sweep_kind", "threshold_lab")
        LAB2_PARENT_RUN_ID = parent.info.run_id
        for thr in threshold_grid:
            for cap in cap_grid:
                with mlflow.start_run(run_name=f"thr{thr}-cap{cap}", nested=True) as child:
                    m = ThresholdAlertModel(score_threshold=thr, amount_cap_usd=cap)
                    preds = m.predict(None, BATCH_DF[["fraud_score", "amount_usd"]])
                    blocked = preds["action"] == "block"
                    captured = float(BATCH_DF.loc[blocked & (BATCH_DF["is_fraud"] == 1), "amount_usd"].sum())
                    false_block = float(BATCH_DF.loc[blocked & (BATCH_DF["is_fraud"] == 0), "amount_usd"].sum())
                    mlflow.log_params({"score_threshold": thr, "amount_cap_usd": cap})
                    mlflow.log_metrics({"captured_fraud_usd": captured, "false_block_usd": false_block})
                    mlflow.set_tag("sweep_kind", "threshold_lab")
                    y_pred = blocked.astype(int).to_numpy()
                    log_confusion_matrix_png(BATCH_DF["is_fraud"].to_numpy(), y_pred, f"thr{thr}-cap{cap}")

    # Scope the search to THIS parent so stale child runs from a prior re-run
    # of the safety-net cannot pollute the winner pick.
    runs_df = mlflow.search_runs(
        experiment_ids=[EXP_ID],
        filter_string=(
            'tags.sweep_kind = "threshold_lab" '
            f'and tags.mlflow.parentRunId = "{LAB2_PARENT_RUN_ID}"'
        ),
        order_by=["metrics.captured_fraud_usd DESC"],
    )
    children = runs_df[runs_df["metrics.captured_fraud_usd"].notna()].copy()
    if children.empty:
        raise RuntimeError("Lab 2 safety-net produced no child runs - check MLflow tracking.")
    children["score"] = children["metrics.captured_fraud_usd"] / (1 + children["metrics.false_block_usd"])
    children = children.sort_values("score", ascending=False)
    winner = children.iloc[0]
    print(f"Winner: score_threshold={winner['params.score_threshold']} amount_cap_usd={winner['params.amount_cap_usd']}")

    with mlflow.start_run(run_name="topic2-lab2-register-winner"):
        winning_model = ThresholdAlertModel(
            score_threshold=float(winner["params.score_threshold"]),
            amount_cap_usd=float(winner["params.amount_cap_usd"]),
        )
        info = mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=winning_model,
            input_example=BATCH_DF[["fraud_score", "amount_usd"]].head(3),
        )
        reg = mlflow.register_model(model_uri=info.model_uri, name=LAB2_REG_NAME)
        client.set_registered_model_alias(name=LAB2_REG_NAME, alias="production", version=reg.version)
        LAB2_PROD_VERSION = reg.version
        print(f"Alias `production` -> version {LAB2_PROD_VERSION}")


## Topic 3 - DVC for local data versioning

MLflow tracks model artifacts beautifully but is awkward for raw data versioning. DVC fills that gap: it stores file content keyed by md5 in a remote (S3 in our case), and writes a tiny `*.dvc` pointer file that goes into git.

- On SageMaker Studio Lab the home directory is a real git workspace - you can `git init`, `dvc init`, and `dvc push` from the System Terminal. We do it from the notebook with `!` shell escapes for convenience.
- The S3 remote uses the execution-role credentials via the standard boto3 credential chain. No extra config inside Studio Lab.
- `dvc.api.read("data/batch.csv")` lets downstream Python read the DVC-tracked file back without explicit S3 commands. That is the reproducibility win.

Docs: `https://dvc.org/doc/user-guide/data-management/remote-storage/amazon-s3`

### Why this matters for Bread Financial

When an auditor asks "what training data produced this model?", `git log data/batch.csv.dvc` gives you the sha-by-sha history of the dataset, and `dvc pull` reconstructs the exact file content. MLflow alone cannot do this for raw inputs.

### Gotchas

- `dvc add` rejects files that are already tracked by git. The demo below creates a fresh `data/` folder so that is not an issue.
- `dvc add` automatically writes the real file into `.gitignore` and creates the `*.dvc` pointer. The pointer is what you commit to git.


In [ ]:
import shutil
from pathlib import Path

# Pre-flight: refuse to start if git or dvc are not on PATH. Otherwise every
# `sh` call returns 127 and the cell limps along until a later read fails
# with a confusing FileNotFoundError on the .dvc pointer.
for tool in ("git", "dvc"):
    if shutil.which(tool) is None:
        raise RuntimeError(
            f"{tool!r} is not on PATH. Install it (or `pip install 'dvc[s3]'` for dvc) "
            "from the System Terminal before running Topic 3."
        )

# Fresh workspace so dvc init is clean
WORKDIR = Path.home() / "week19_optional_dvc_demo"
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True)
os.chdir(WORKDIR)

def sh(cmd, check=True):
    """Tiny shell helper. By default raises on nonzero exit so silent failures
    cannot snowball into a confusing error three cells later. Pass check=False
    for commands where failure is informational (e.g. `ls`)."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout.strip())
    if r.returncode != 0 and r.stderr: print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed (exit {r.returncode}): {cmd}")
    return r.returncode

# git + dvc init
sh("git init -q")
sh("git config user.email week19@bread.local")
sh("git config user.name 'Week19 Student'")
sh("dvc init -q")
sh("git add .dvc .dvcignore && git commit -qm 'dvc init'")

# Materialize BATCH_DF from Topic 2 as a local CSV
(WORKDIR / "data").mkdir()
BATCH_DF.to_csv(WORKDIR / "data" / "batch.csv", index=False)

# dvc add -> creates data/batch.csv.dvc pointer + writes real file to .gitignore
sh("dvc add data/batch.csv")
sh("git add data/.gitignore data/batch.csv.dvc && git commit -qm 'dvc track batch.csv'")

# dvc remote add (S3) and push
DVC_REMOTE_URL = f"s3://{S3_BUCKET}/{S3_PREFIX}/dvc-store"
sh(f"dvc remote add -d s3-remote {DVC_REMOTE_URL}")
sh("git add .dvc/config && git commit -qm 'dvc remote -> s3'")
sh("dvc push")

print("\nFiles in repo:")
sh("ls -la data/", check=False)
print("\nThe pointer file content (this is what goes in git):")
print((WORKDIR / "data" / "batch.csv.dvc").read_text())


### Lab 3: Read the DVC-tracked batch back with dvc.api

Steps:

1. Inside the same `WORKDIR` repo, modify `data/batch.csv` (e.g. append 5 rows so it now has 105 rows). Run `dvc add data/batch.csv` again. Commit the updated pointer to git. Run `dvc push`. You now have two versions of the dataset, distinguished by md5 in `data/batch.csv.dvc`.
2. Note the new md5 hash printed by `dvc add`.
3. Use `dvc.api.read("data/batch.csv", repo=str(WORKDIR), mode="rb")` to read the file back as bytes, then `pd.read_csv(io.BytesIO(...))` to confirm it has 105 rows. Set `LAB3_NEW_LEN`.
4. Then `git checkout HEAD~1 -- data/batch.csv.dvc && dvc pull data/batch.csv` to roll back to the 100-row version. Read it again and set `LAB3_OLD_LEN = 100`.
5. Restore `HEAD` so the rest of the notebook stays in a clean state.

**Success criterion**: you printed both 105 and 100 for the same logical file path, by switching only the `.dvc` pointer in git.


In [ ]:
import io
import dvc.api

# YOUR CODE
# 1. Append 5 rows to data/batch.csv (use BATCH_DF.head() to fabricate).
# 2. dvc add again, git commit, dvc push.

LAB3_NEW_LEN = None
LAB3_OLD_LEN = None

# YOUR CODE
# 3. Read the current version via dvc.api.read and confirm length.

# YOUR CODE
# 4. Roll back the pointer with git checkout HEAD~1, dvc pull, re-read, confirm length.

print(f"Current version rows: {LAB3_NEW_LEN}")
print(f"Rolled-back version rows: {LAB3_OLD_LEN}")


In [ ]:
# Lab 3 SAFETY-NET: only runs if LAB3_NEW_LEN or LAB3_OLD_LEN are not set.
if LAB3_NEW_LEN is None or LAB3_OLD_LEN is None:
    print("Using Lab 3 safety-net.")
    os.chdir(WORKDIR)
    extra = BATCH_DF.head(5)
    cur = pd.read_csv(WORKDIR / "data" / "batch.csv")
    pd.concat([cur, extra], ignore_index=True).to_csv(WORKDIR / "data" / "batch.csv", index=False)
    sh("dvc add data/batch.csv")
    sh("git add data/batch.csv.dvc && git commit -qm 'dvc: batch +5 rows'")
    sh("dvc push")

    raw = dvc.api.read("data/batch.csv", repo=str(WORKDIR), mode="rb")
    LAB3_NEW_LEN = len(pd.read_csv(io.BytesIO(raw)))

    sh("git checkout HEAD~1 -- data/batch.csv.dvc")
    sh("dvc pull data/batch.csv")
    LAB3_OLD_LEN = len(pd.read_csv(WORKDIR / "data" / "batch.csv"))

    # Restore HEAD pointer so the rest of the notebook is in a clean state
    sh("git checkout HEAD -- data/batch.csv.dvc")
    sh("dvc pull data/batch.csv")

    print(f"Current version rows: {LAB3_NEW_LEN}")
    print(f"Rolled-back version rows: {LAB3_OLD_LEN}")


## Topic 4 - Lineage and SageMaker Pipelines preview

Production question: "where did this deployed model come from?" The answer links Model Package -> Training Job -> data version (DVC md5) -> git commit.

`sagemaker_client.describe_model_package(ModelPackageName=arn)` returns package metadata, but the training job ARN is NOT a top-level field. Two ways to recover it:

- **Easy** (what we demo): store it yourself in `CustomerMetadataProperties` (a string-to-string dict) when calling `create_model_package`. Then `describe_model_package` returns it under the same key.
- **Advanced**: `sagemaker_client.list_associations(DestinationArn=model_package_arn)` returns AssociationSummaries linking `SourceArn` to `DestinationArn`. SageMaker auto-populates this when the package was registered from a Pipeline.

For MLflow runs the lineage is bidirectional: the model package gets an `mlflow_run_id` tag in `CustomerMetadataProperties`, and the MLflow run gets a `model_package_arn` tag via `mlflow.set_tag()`. With both sides linked, an auditor can start at either end and reach the other.

### SageMaker Pipelines as CI/CD

In real production the Week 19 training job is not run by hand - it lives inside a SageMaker Pipeline that gets re-run on every PR merge:

```
   ProcessingStep (prep data)
        |
        v
   TrainingStep <-- this is the Estimator.fit() from the main Week 19 notebook
        |
        v
   ConditionStep (if accuracy > 0.85)
        |
        +--> RegisterModel (create_model_package + set alias)
        |
        +--> (else) FailStep
```

Each `pipeline.start()` becomes a Pipeline Execution. The `pipeline_execution_arn` is the natural CI/CD unit: rerun on PR merge, gate registration on quality, and get full lineage for free because SageMaker auto-creates Associations between every step's outputs and inputs.

Docs: `https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines.html`

We will BUILD a pipeline definition below to see its shape, but we will NOT call `pipeline.upsert()` or `pipeline.start()` - that would incur charges on the class account.


In [ ]:
# Find a recent training job to attach lineage to.
recent = sagemaker_client.list_training_jobs(
    SortBy="CreationTime", SortOrder="Descending", MaxResults=5,
)["TrainingJobSummaries"]
if not recent:
    print("No training jobs found - run the main Week 19 notebook first.")
    raise RuntimeError("No training job")

MAIN_TRAINING_JOB_NAME = recent[0]["TrainingJobName"]
MAIN_TRAINING_JOB_ARN  = recent[0]["TrainingJobArn"]
print(f"Using training job: {MAIN_TRAINING_JOB_NAME}")

# Read the DVC md5 directly from the .dvc pointer file (this is the data version)
import yaml
dvc_pointer = yaml.safe_load((WORKDIR / "data" / "batch.csv.dvc").read_text())
DATA_VERSION_MD5 = dvc_pointer["outs"][0]["md5"]
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR).decode().strip()[:9]
print(f"data md5:   {DATA_VERSION_MD5}")
print(f"git commit: {GIT_COMMIT}")

with mlflow.start_run(run_name="topic4-demo-lineage") as run:
    # Tag the MLflow run with the lineage so the link is bidirectional
    mlflow.set_tag("git_commit",        GIT_COMMIT)
    mlflow.set_tag("data_version_md5",  DATA_VERSION_MD5)
    mlflow.set_tag("training_job_arn",  MAIN_TRAINING_JOB_ARN)

    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ThresholdAlertModel(),
        input_example=BATCH_DF[["fraud_score", "amount_usd"]].head(3),
    )

    pkg_group = f"week19-optional-demo-{USER_ID}"
    try:
        sagemaker_client.create_model_package_group(ModelPackageGroupName=pkg_group)
    except sagemaker_client.exceptions.ResourceInUse:
        pass

    pkg_response = sagemaker_client.create_model_package(
        ModelPackageGroupName=pkg_group,
        ModelPackageDescription=f"Demo threshold model, MLflow run {run.info.run_id}",
        InferenceSpecification={
            "Containers": [{
                "Image": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-inference:2.1.0-transformers4.37.0-cpu-py310-ubuntu22.04",
                "ModelDataUrl": f"s3://{S3_BUCKET}/{S3_PREFIX}/placeholder-model.tar.gz",
            }],
            "SupportedContentTypes": ["application/json"],
            "SupportedResponseMIMETypes": ["application/json"],
        },
        CustomerMetadataProperties={
            "mlflow_run_id":      run.info.run_id,
            "training_job_arn":   MAIN_TRAINING_JOB_ARN,
            "git_commit":         GIT_COMMIT,
            "data_version_md5":   DATA_VERSION_MD5,
        },
        ModelApprovalStatus="PendingManualApproval",
    )
    DEMO_PACKAGE_ARN = pkg_response["ModelPackageArn"]
    mlflow.set_tag("model_package_arn", DEMO_PACKAGE_ARN)
    print(f"Registered: {DEMO_PACKAGE_ARN}")

# Round-trip via the easy path: CustomerMetadataProperties
detail = sagemaker_client.describe_model_package(ModelPackageName=DEMO_PACKAGE_ARN)
print("\nLineage from CustomerMetadataProperties:")
for k, v in detail.get("CustomerMetadataProperties", {}).items():
    print(f"  {k:20s} {v}")

# Advanced path preview: list_associations. Often empty for manually-registered
# packages, but populated when registered from a SageMaker Pipeline.
assoc = sagemaker_client.list_associations(DestinationArn=DEMO_PACKAGE_ARN).get("AssociationSummaries", [])
print(f"\nlist_associations returned {len(assoc)} entries (often 0 for manual registration):")
for a in assoc:
    print(f"  {a.get('AssociationType')}: {a.get('SourceType')} -> {a.get('DestinationType')}")


In [ ]:
# DEMO ONLY - we BUILD the pipeline definition but do NOT call pipeline.start().
# This shows what the Week 19 training job looks like as a CI/CD pipeline step.

from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterFloat, ParameterString
from sagemaker.workflow.steps import TrainingStep
from sagemaker.estimator import Estimator

pipeline_session = PipelineSession()

# Parameters become inputs to pipeline.start() so the same definition can run
# against different data or accuracy bars without code edits.
accuracy_threshold = ParameterFloat(name="AccuracyThreshold", default_value=0.85)
input_data         = ParameterString(name="InputData", default_value=f"s3://{S3_BUCKET}/{S3_PREFIX}/")

# Placeholder Estimator - in real life this is the same Estimator object from the
# main Week 19 notebook (sklearn / xgboost / huggingface).
training_estimator = Estimator(
    image_uri="763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.1.0-transformers4.36.0-gpu-py310-cu121-ubuntu20.04",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=pipeline_session,
)

step_train = TrainingStep(name="TrainModel", estimator=training_estimator)

# Note: in a real pipeline we would add ConditionStep + ModelStep here. We
# deliberately do NOT construct ModelStep in this demo because ModelStep
# requires step_args produced by model.register(...) - passing a placeholder
# raises at construction time. The diagram in the markdown above shows the
# full topology; the homework extension at the bottom of the notebook walks
# through registering a real Model and wiring step_register correctly.
#
# Pseudo-code for the missing pieces (do NOT uncomment without a real Model):
#
#   from sagemaker.workflow.condition_step import ConditionStep
#   from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
#   from sagemaker.workflow.fail_step import FailStep
#   from sagemaker.workflow.functions import JsonGet
#   from sagemaker.workflow.model_step import ModelStep
#
#   register_args = my_model.register(
#       content_types=["application/json"],
#       response_types=["application/json"],
#       inference_instances=["ml.m5.large"],
#       transform_instances=["ml.m5.large"],
#       model_package_group_name="my-group",
#   )
#   step_register = ModelStep(name="RegisterModel", step_args=register_args)
#   step_fail     = FailStep(name="FailOnLowAccuracy",
#                            error_message="Accuracy below threshold")
#   cond          = ConditionGreaterThanOrEqualTo(
#       left=JsonGet(step_name=step_train.name, property_file="metrics",
#                    json_path="accuracy"),
#       right=accuracy_threshold,
#   )
#   step_cond     = ConditionStep(name="GateOnAccuracy",
#                                 conditions=[cond],
#                                 if_steps=[step_register],
#                                 else_steps=[step_fail])

pipeline = Pipeline(
    name=f"week19-optional-pipeline-demo-{USER_ID}",
    parameters=[accuracy_threshold, input_data],
    steps=[step_train],  # only the training step for the demo
    sagemaker_session=pipeline_session,
)

print("Pipeline definition (NOT submitted):")
print(json.dumps(json.loads(pipeline.definition()), indent=2)[:1200] + "\n...")
print("\nTo actually deploy: pipeline.upsert(role_arn=role); pipeline.start()")
print("That call is intentionally NOT made here to avoid charging the class account.")


### Lab 4: Build a full lineage lookup function

Write a function `trace_model(package_arn: str) -> dict` that, given a Model Package ARN, returns a dict with these keys:

- `package_arn`: the input ARN.
- `mlflow_run_id`: from `CustomerMetadataProperties`.
- `training_job_arn`: from `CustomerMetadataProperties`.
- `git_commit`: from `CustomerMetadataProperties`.
- `data_version_md5`: from `CustomerMetadataProperties`.
- `training_job_status`: by calling `describe_training_job` on the training job NAME (parse the name from the ARN - it is the last `/`-separated segment).
- `mlflow_run_tags`: dict of tag name to value, via `mlflow.get_run(mlflow_run_id).data.tags`.
- `associations`: a list of `(SourceType, SourceArn)` tuples from `list_associations(DestinationArn=package_arn)`.

Then call `trace_model(DEMO_PACKAGE_ARN)` and pretty-print the result.

**Success criterion**: every key has a non-None value (associations may be an empty list - that is OK for manual registration) and `git_commit` matches the one set in the demo cell above.


In [ ]:
def trace_model(package_arn: str) -> dict:
    # YOUR CODE
    return {}

result = trace_model(DEMO_PACKAGE_ARN)
print(json.dumps(result, indent=2, default=str))


## Think About It (cross-topic) and Homework Extensions

### Think About It

Reflection questions - no group activity, just self-reflection:

1. The threshold pyfunc in Topic 1 is stateless: the params live on the instance. The Strands pyfunc in Lab 1 has live state (the agent). Which one is safer to serve in a multi-replica SageMaker endpoint, and why?
2. In Topic 2 you assigned the `production` alias. If you ran the sweep again next week and got a better f1, what is the safe sequence of operations to swap traffic, and how does this differ from the old "stages" model?
3. In Topic 3 you tracked `data/batch.csv` with DVC. If the S3 remote was accidentally deleted, what survives? What is the minimum you would need from backups to rebuild the lineage?
4. In Topic 4 your `trace_model` reads `CustomerMetadataProperties` AND `list_associations`. Which one would survive longer in a real audit scenario where the training job was deleted after 90 days?

### Homework Extensions (async, no rubric)

- **Topic 1**: Serve the threshold pyfunc behind a SageMaker Serverless endpoint using `mlflow.sagemaker.deploy()`. Try it with the same Lab 1 batch.
- **Topic 2**: Convert the threshold sweep from a nested loop to Optuna with the MLflow Optuna callback. Compare the resulting run tree in the UI.
- **Topic 3**: Add a second DVC-tracked file (e.g. a `params.yaml`) and use `dvc.api.params_show()` to read its current values from any commit.
- **Topic 4**: Actually submit the pipeline above (`pipeline.upsert()` + `pipeline.start()`) with a minimal training script and watch the Associations table populate automatically. Re-run `trace_model` against the resulting Model Package and observe how `associations` is now non-empty.
